### Deepseek : Next Token Prediction and Perplexity

In [1]:
import math
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
).to(device)
model.eval()

C:\Users\Dell\OneDrive\Documents\TomDocuments\PythonProjects\DL\20261001\DeepLearning\DL_MATH70116_2026\dl_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:03<00:00, 103.63it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

### Next token prediction

In [2]:
def top_next_tokens(text, k=10):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits          # (1, seq_len, vocab)
    probs = F.softmax(logits[0, -1].float(), dim=-1)   # distribution for the next token
    top = torch.topk(probs, k)
    return [(tokenizer.decode([int(i)]), p.item())
            for p, i in zip(top.values, top.indices)]

In [3]:
for token, prob in top_next_tokens("The animal did not cross the road because it was too", k=20):
    print(f"{token!r:>15}  {prob:.4f}")

        ' slow'  0.1896
        ' late'  0.1644
        ' long'  0.0719
         ' far'  0.0679
        ' busy'  0.0546
       ' close'  0.0471
       ' short'  0.0443
     ' nervous'  0.0385
        ' fast'  0.0300
       ' tired'  0.0194
       ' small'  0.0118
     ' crowded'  0.0099
        ' tall'  0.0088
       ' heavy'  0.0086
       ' early'  0.0081
         ' low'  0.0056
   ' dangerous'  0.0050
   ' difficult'  0.0050
         ' shy'  0.0047
         ' big'  0.0044


<b>Exercise:</b> Rewrite the method <code>top_next_tokens</code> so that it outputs the least likely tokens

### Perplexity

In [4]:
def perplexity(text, show_tokens=False):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    ids = inputs["input_ids"]
    with torch.no_grad():
        logits = model(**inputs).logits.float()

    # The logits at position i predict token i+1
    log_probs = F.log_softmax(logits[0, :-1], dim=-1)
    targets = ids[0, 1:]
    token_lp = log_probs.gather(1, targets.unsqueeze(1)).squeeze(1)   # log p(x_i | x_<i)

    nll = -token_lp.mean().item()
    if show_tokens:
        for t, lp in zip(targets, token_lp):
            print(f"{tokenizer.decode([int(t)])!r:>15}  p={math.exp(lp.item()):.4f}")
    return math.exp(nll)

In [5]:
print(perplexity("The animal did not cross the road because it was too tired.", show_tokens=True))
print(perplexity("The animal did not cross the road because it was too slow.", show_tokens=True))
print(perplexity("Road the tired not animal because was it cross too the did.", show_tokens=True))

      ' animal'  p=0.0001
         ' did'  p=0.0000
         ' not'  p=0.4321
       ' cross'  p=0.0070
         ' the'  p=0.3870
        ' road'  p=0.0460
     ' because'  p=0.0369
          ' it'  p=0.3076
         ' was'  p=0.4249
         ' too'  p=0.2250
       ' tired'  p=0.0194
            '.'  p=0.2421
33.9561245467498
      ' animal'  p=0.0001
         ' did'  p=0.0000
         ' not'  p=0.4321
       ' cross'  p=0.0070
         ' the'  p=0.3870
        ' road'  p=0.0460
     ' because'  p=0.0369
          ' it'  p=0.3076
         ' was'  p=0.4249
         ' too'  p=0.2250
        ' slow'  p=0.1896
            '.'  p=0.2293
28.20963944913393
         ' the'  p=0.0047
       ' tired'  p=0.0000
         ' not'  p=0.0000
      ' animal'  p=0.0000
     ' because'  p=0.0004
         ' was'  p=0.0004
          ' it'  p=0.0063
       ' cross'  p=0.0001
         ' too'  p=0.0001
         ' the'  p=0.0013
         ' did'  p=0.0003
            '.'  p=0.0145
3898.0450539872118


In [6]:
def bottom_next_tokens(text, k=10):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = F.softmax(logits[0, -1].float(), dim=-1)
    bottom = torch.topk(probs, k, largest=False)
    return [(tokenizer.decode([int(i)]), p.item())
            for p, i in zip(bottom.values, bottom.indices)]

In [7]:
for token, prob in bottom_next_tokens("The animal did not cross the road because it was too", k=20):
    print(f"{token!r:>15}  {prob:.4f}")

            '更'  0.0000
            '比'  0.0000
      ' Finite'  0.0000
      ' Depths'  0.0000
  ' resurgence'  0.0000
  ' negativity'  0.0000
    ' Deleting'  0.0000
            '既'  0.0000
         ' Dot'  0.0000
         ' Dou'  0.0000
     ' Highest'  0.0000
    ' Gameplay'  0.0000
          '是不是'  0.0000
    ' Composer'  0.0000
            '以'  0.0000
  ' Generating'  0.0000
      'quality'  0.0000
     ' deletes'  0.0000
      'editing'  0.0000
   ' Promotion'  0.0000
